# 03 RF Load and Current Budget

**Updated for locked hybrid geometry: 500 µm added-pair setback.**

This notebook uses the full 2×2 differential capacitance matrix from the hybrid
electrode FEM model, not a scalar capacitance approximation.

Geometry:
- z-cut LiNbO3, 3 mm × 3 mm × 30 mm
- Full-face +x/-x electrodes (existing pair)
- 2 mm centered strip +y/-y electrodes (added pair, 500 µm setback)
- C_diff_xx ≠ C_diff_yy (asymmetric channels)

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rot_eo_model.rf_lumped import (
    cap_current_peak_A, cap_apparent_power_VA,
    dielectric_loss_W, z_capacitor, reflected_capacitance_F
)

# ── Load locked geometry spec from JSON ──
spec_path = ROOT / "rf_ready_geometry_spec.json"
with open(spec_path) as f:
    spec = json.load(f)

# Differential capacitance matrix (2x2, in Farads)
C_diff = np.array(spec['C_diff_pF']) * 1e-12
C_xx = C_diff[0, 0]
C_yy = C_diff[1, 1]
C_xy = C_diff[0, 1]

# Half-wave voltages (asymmetric)
Vpi_x = spec['Vpi_x_V']
Vpi_y = spec['Vpi_y_V']

# Field-per-volt matrix
A = np.array(spec['A_V_per_m_per_V'])

print(f"Locked geometry: {spec['geometry']['strip_setback_um']} um setback")
print(f"C_diff_xx = {C_xx*1e12:.3f} pF (full-face channel)")
print(f"C_diff_yy = {C_yy*1e12:.3f} pF (strip channel)")
print(f"C_diff_xy = {C_xy*1e12:.4f} pF (cross-coupling)")
print(f"Vpi_x = {Vpi_x:.1f} V, Vpi_y = {Vpi_y:.1f} V")
print(f"Amplitude ratio Vpi_y/Vpi_x = {Vpi_y/Vpi_x:.4f}")

## Single-Channel Current vs Frequency

For each channel independently: $I_{peak} = \omega \, C_{diff,ii} \, V_{\pi,i}$

In [ ]:
freqs = np.linspace(6e6, 102e6, 500)

Ipk_x = cap_current_peak_A(freqs, C_xx, Vpi_x)
Ipk_y = cap_current_peak_A(freqs, C_yy, Vpi_y)

plt.figure(figsize=(9, 5))
plt.plot(freqs/1e6, Ipk_x, label=f'x-channel: C={C_xx*1e12:.1f} pF, V={Vpi_x:.0f} V', linewidth=2)
plt.plot(freqs/1e6, Ipk_y, label=f'y-channel: C={C_yy*1e12:.1f} pF, V={Vpi_y:.0f} V', linewidth=2)
plt.xlabel('Frequency [MHz]')
plt.ylabel('Peak current [A]')
plt.title('EO Capacitor Current: Asymmetric Hybrid Geometry')
plt.legend()
plt.grid(True)
plt.tight_layout()

## Summary Table: Both Channels

In [ ]:
summary_freqs = np.array([6e6, 10e6, 30e6, 50e6, 100e6])

df = pd.DataFrame({
    'f_MHz': summary_freqs/1e6,
    'Xc_x_ohm': np.abs(z_capacitor(summary_freqs, C_xx)),
    'Xc_y_ohm': np.abs(z_capacitor(summary_freqs, C_yy)),
    'Ipeak_x_A': cap_current_peak_A(summary_freqs, C_xx, Vpi_x),
    'Ipeak_y_A': cap_current_peak_A(summary_freqs, C_yy, Vpi_y),
    'VAR_x': cap_apparent_power_VA(summary_freqs, C_xx, Vpi_x),
    'VAR_y': cap_apparent_power_VA(summary_freqs, C_yy, Vpi_y),
})
df

## Quadrature Phasor Current

For simultaneous quadrature drive $V_{diff} = [V_{\pi,x},\; -jV_{\pi,y}]$:

$$I_{diff}(\omega) = j\omega\, C_{diff} \cdot V_{diff}$$

$$|I_x| = \omega \sqrt{(C_{xx} V_{\pi,x})^2 + (C_{xy} V_{\pi,y})^2}$$
$$|I_y| = \omega \sqrt{(C_{xy} V_{\pi,x})^2 + (C_{yy} V_{\pi,y})^2}$$

In [ ]:
omega = 2 * np.pi * summary_freqs

Ix_quad = omega * np.sqrt((C_xx * Vpi_x)**2 + (C_xy * Vpi_y)**2)
Iy_quad = omega * np.sqrt((C_xy * Vpi_x)**2 + (C_yy * Vpi_y)**2)

df_quad = pd.DataFrame({
    'f_MHz': summary_freqs / 1e6,
    'Ix_quad_A': Ix_quad,
    'Iy_quad_A': Iy_quad,
    'max_I_A': np.maximum(Ix_quad, Iy_quad),
})
print("Quadrature drive peak currents:")
df_quad

## Transformer Step-Up: Reflected Capacitance

Each channel needs its own transformer design since $C_{xx} \neq C_{yy}$.

In [ ]:
print("X-channel (full-face, C_xx):")
for N in [1, 1.5, 2, 3]:
    print(f"  Step-up N={N}: primary sees C = {reflected_capacitance_F(C_xx, N)*1e12:.1f} pF")

print(f"\nY-channel (strip, C_yy):")
for N in [1, 1.5, 2, 3]:
    print(f"  Step-up N={N}: primary sees C = {reflected_capacitance_F(C_yy, N)*1e12:.1f} pF")

## Dielectric Loss Sensitivity

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

tan_deltas = [1e-4, 5e-4, 1e-3, 5e-3]
for td in tan_deltas:
    Px = dielectric_loss_W(freqs, C_xx, Vpi_x, td)
    ax1.plot(freqs/1e6, Px, label=f'tan_d={td:g}')
ax1.set_xlabel('Frequency [MHz]')
ax1.set_ylabel('Dielectric loss [W]')
ax1.set_title(f'X-channel loss (C={C_xx*1e12:.1f} pF, V={Vpi_x:.0f} V)')
ax1.legend(); ax1.grid(True)

for td in tan_deltas:
    Py = dielectric_loss_W(freqs, C_yy, Vpi_y, td)
    ax2.plot(freqs/1e6, Py, label=f'tan_d={td:g}')
ax2.set_xlabel('Frequency [MHz]')
ax2.set_ylabel('Dielectric loss [W]')
ax2.set_title(f'Y-channel loss (C={C_yy*1e12:.1f} pF, V={Vpi_y:.0f} V)')
ax2.legend(); ax2.grid(True)

fig.tight_layout()

## Key Takeaways

1. The y-channel (strip electrode) is the bottleneck: higher $V_\pi$ and thus higher current.
2. Cross-coupling $C_{xy} \approx 0$ means the channels are essentially independent for RF design.
3. Each channel needs its own RF matching network with different $C$ and $V_\pi$ targets.
4. The RF PA must handle the y-channel current at the highest frequency of interest.

Change `C_xx`, `C_yy`, `Vpi_x`, `Vpi_y` aggressively if exploring alternative geometries.
Every added pF at 100 MHz and hundreds of volts matters.